In [ ]:
# import libraries
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
from numpy.random import seed
from tensorflow.random import set_seed
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import math

# set random seedindexes_AC1
seed(10)
set_seed(10)

training_dataset_name = "run53_mix_mega_shared"
testing_dataset_name = "run57_mix_mega_shared"

val_number = 6000
# If not GPU is available, set the following to -1
os.environ["CUDA_VISIBLE_DEVICES"]="0"
numpy_file=0

number_of_detectors = 6
data_path = "/data/test_newrepo/"

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:

# load training dataset
file_path = data_path+'/'+training_dataset_name+'_dataset.pkl'

with open(file_path, 'rb') as file:
    training_dataset_array = pickle.load(file)

# load testing dataset
file_path = data_path+'/'+testing_dataset_name+'_dataset.pkl'

with open(file_path, 'rb') as file:
    testing_dataset_array = pickle.load(file)

In [ ]:
def diff_phi(a1, a2):
    diff = abs(a1 - a2)
    if diff > 180:
        diff = 360 - diff
    return diff

def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg


def l2_normalize(data):
  """
  Normalize a NumPy array using the L2 (Euclidean) norm.

  Args:
    data (numpy.ndarray): The array to normalize. Can be a 1D vector
                         or a 2D matrix (where each row is a vector to normalize).

  Returns:
    numpy.ndarray: The L2-normalized array.
  """

  data = np.array(data)
  if data.ndim == 1:
    # Case: 1D vector
    norm = np.sqrt(np.sum(data**2))
    if norm == 0:
      return data  # Avoid division by zero if the vector is null
    return data / norm
  elif data.ndim == 2:
    # Case: 2D matrix (normalize each row)
    norms = np.sqrt(np.sum(data**2, axis=1, keepdims=True))
    # Handle the case of null norms (zero rows)
    norms[norms == 0] = 1
    return data / norms
  else:
    raise ValueError("Input must be a 1D or 2D array.")

    
def prepare_dataset(data_array):
  """
  Prepare the dataset and corresponding labels for training or evaluation.

  Steps performed:
    1. Normalize the detector counts using L2 normalization.
    2. Extract longitude and latitude labels.
    3. Optionally scale labels between 0 and 1 (if `normalize == 1`).
    4. Convert labels to radians for further processing.

  Args:
    data_array (list of dict): Each element must contain:
        - 'counts' (numpy.ndarray): Detector count values.
        - 'coord' (tuple or list): Coordinates (longitude, latitude).

  Returns:
    tuple:
      - dataset (numpy.ndarray): Normalized detector counts.
      - labels (numpy.ndarray): Original labels (longitude, latitude).
      - labels_norm (numpy.ndarray): Scaled labels (if normalization applied).
      - theta (tuple): Extracted theta values in degrees.
      - phi (tuple): Extracted phi values in degrees.
      - coords_rad (list): List of coordinates in radians.
  """

  dataset = np.empty((len(data_array), number_of_detectors))
  labels = np.empty((len(data_array), 2))
    
  count = -1
  for element in data_array:
      count += 1
      dataset[count] = l2_normalize(element['counts'])
      labels[count][0] = float(element['coord'][0])
      labels[count][1] = float(element['coord'][1])
        
  labels = labels[:count+1]
  dataset = dataset[:count+1]

    
  labels_norm = np.empty((len(dataset), 2))
    
  count = -1
  for element in labels:
      count += 1
      labels_norm[count][0] = labels[count][0]
      labels_norm[count][1] = labels[count][1]

  # Convert to radians
  coords_rad = []
    
  for theta, phi in labels:
      coords_rad.append([theta, phi])

  theta, phi = zip(*coords_rad)
    
  return dataset, labels, labels_norm, theta, phi, coords_rad



In [ ]:
# prepare training and validation dataset
dataset, labels, labels_norm, theta, phi, coords_rad = prepare_dataset(training_dataset_array)

In [ ]:
dataset.shape

In [ ]:
# prepare testing and validation dataset
test_dataset, test_labels, test_labels_norm, test_theta, test_phi, test_coords_rad = prepare_dataset(testing_dataset_array)

In [ ]:
test_dataset.shape

In [ ]:
#split val and test dataset
random_indices = np.random.permutation(len(dataset))

N = dataset.shape[0]
validation_indices = random_indices[:val_number]
train_indices = np.setdiff1d(np.arange(N), validation_indices)

validation_dataset = dataset[validation_indices]
training_dataset = dataset[train_indices]

validation_labels = labels[validation_indices]
training_labels = labels[train_indices]


In [ ]:

coord_array = np.array(coords_rad)
coord_array.shape

In [ ]:
plt.hist(coord_array[:, 0])
plt.xlabel("Theta")
plt.ylabel("Counts")

In [ ]:
plt.hist(coord_array[:, 1])
plt.xlabel("Phi")
plt.ylabel("Counts")

In [ ]:
# Define the model architecture
model = keras.Sequential([
    keras.layers.Dense(128*2, input_shape=(number_of_detectors,)), # activation='relu',#,kernel_regularizer=tf.keras.regularizers.l2(0.01)
    keras.layers.LeakyReLU(alpha=0.1) ,
    keras.layers.Dropout(0.02),
    keras.layers.Dense(64*2), # , activation='relu'#,kernel_regularizer=tf.keras.regularizers.l2(0.01)
    keras.layers.LeakyReLU(alpha=0.1),
    keras.layers.Dropout(0.02),
    keras.layers.Dense(32*2), #activation='relu'
    keras.layers.LeakyReLU(alpha=0.1),
    keras.layers.Dropout(0.02),
    keras.layers.Dense(2)
])

# Compile the model
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")
model.summary()

In [ ]:
history = model.fit(
    x=training_dataset,
    y=training_labels,
    epochs=2000,
    batch_size=256,
    validation_data=(validation_dataset,validation_labels),
    validation_split=0.2,
    shuffle=True,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, mode="min")
    ],
)


In [ ]:
fig, ax1 = plt.subplots(1, 1,figsize=(10,5))
fig.suptitle('Training')
      
ax1.plot(history.history["loss"], label="Training Loss")
ax1.plot(history.history["val_loss"], label="Validation Loss")
    
ax1.legend()
plt.show() 

In [ ]:
#save model for inference when it is ready
save_model = False
if(save_model):
    model_name = training_dataset_name+"_simple_model.keras"
    model.save(data_path+"/"+model_name)

In [ ]:
#Perform prediction on the test dataset
pred_data = model.predict(test_dataset)
   

In [ ]:
plt.hist(test_labels[:, 0],bins=50,alpha=0.5)
plt.hist(pred_data[:, 0],bins=50,alpha=0.5)
plt.xlabel("Theta")
plt.ylabel("Counts")

In [ ]:
plt.hist(test_labels[:, 1],bins=50,alpha=0.5)
plt.hist(pred_data[:, 1],bins=50,alpha=0.5)
plt.xlabel("Phi")
plt.ylabel("Counts")

In [ ]:
max_lon = 180.0
min_lon = 0.0

max_lat = 360.0
min_lat = 0.0

pred_data_original = np.empty((len(pred_data),2))

pred_data_original[:, 0] = pred_data[:, 0]
pred_data_original[:, 1] = pred_data[:, 1]

test_labels_original = np.empty((len(test_labels),2))

test_labels_original[:, 0] = test_labels[:, 0]
test_labels_original[:, 1] = test_labels[:, 1]

absolute_diffs_theta = np.abs(pred_data_original[:, 0] - test_labels_original[:, 0])
absolute_diffs_phi = np.abs(pred_data_original[:, 1] - test_labels_original[:, 1])

# Calculate the Mean Absolute Error (MAE) for each element separately
mae_theta = np.mean(absolute_diffs_theta)
mae_phi = np.mean(absolute_diffs_phi)

print("Mean Absolute Error for Theta:", mae_theta)
print("Mean Absolute Error for Phi:", mae_phi)

In [ ]:
# Calculate distances between corresponding coordinates

distances = []
theta_distances = []
phi_distances = []

for i in range(0,len(pred_data_original)):

    d = angular_distance(pred_data_original[i][0],pred_data_original[i][1],test_labels_original[i][0],test_labels_original[i][1])
    theta_dist = np.abs(pred_data_original[i][0]-test_labels_original[i][0])
    phi_dist = diff_phi(pred_data_original[i][1],test_labels_original[i][1])
    distances.append(d)
    theta_distances.append(theta_dist)
    phi_distances.append(phi_dist)


In [ ]:
print(np.mean(distances))
print(np.mean(theta_distances))
print(np.mean(phi_distances))

In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
hp.projview(
    np.array(distances),
    coord=["G"],
    projection_type="aitoff",          
    graticule=True,
    graticule_labels=True,
    longitude_grid_spacing=60,
    title=file,
    latitude_grid_spacing=30,
    cmap="turbo",
    nest=True,
    unit="", 
    fontsize={
        "xlabel": 14,
        "ylabel": 14,
        "title": 16,
        "xtick_label": 14,
        "ytick_label": 14,
        "cbar_label": 14,
        "cbar_tick_label": 14 
    },
    override_plot_properties={
        "cbar_shrink": 0.8,
        "cbar_pad": 0.05,
        "cbar_label_pad": 5
    }
    
)
plt.show()

In [ ]:
if False:

    # Salva l'array in un file usando pickle
    with open(data_path+"/"+testing_dataset_name+"_distances_simple_model.pkl", "wb") as f:
        pickle.dump(distances, f)

    # Salva l'array in un file usando pickle
    with open(data_path+"/"+testing_dataset_name+"_conf_area_simple_model.pkl", "wb") as f:
        pickle.dump(conf_area, f)
